In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: PREPARING BENCHMARK AND OFFICIAL AF-CLIP ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
AFCLIP_ROOT = Path("/kaggle/working/AF-CLIP")
AFCLIP_COMMIT = "bb7edec4128a76f29cb573cd3002538bf250b2fe"

if not BENCHMARK_ROOT.exists():
    subprocess.run(["git", "clone", f"https://github.com/{BENCHMARK_REPOSITORY}.git", str(BENCHMARK_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(BENCHMARK_ROOT), "pull", "--ff-only"], check=True)
required_model_file = BENCHMARK_ROOT / "few_shot" / "harness" / "models.py"
if not required_model_file.is_file() or "AFCLIPFewShotWrapper" not in required_model_file.read_text(encoding="utf-8"):
    raise RuntimeError("The cloned benchmark revision does not contain AF-CLIP+ few-shot support. Commit and push these changes first.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", str(BENCHMARK_ROOT / "few_shot" / "requirements.txt")], check=True)
if not AFCLIP_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/Faustinaqq/AF-CLIP.git", str(AFCLIP_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(AFCLIP_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(AFCLIP_ROOT), "checkout", "--detach", AFCLIP_COMMIT], check=True)
resolved_commit = subprocess.run(["git", "-C", str(AFCLIP_ROOT), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
if resolved_commit != AFCLIP_COMMIT:
    raise RuntimeError(f"Wrong AF-CLIP source commit: {resolved_commit}")

for import_path in (BENCHMARK_ROOT, AFCLIP_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["AFCLIP_ROOT"] = str(AFCLIP_ROOT)

import numpy as np
import torch
from PIL import Image
from shared.corruption import apply_corruption

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle before running AF-CLIP+.")
smoke_image = Image.fromarray(np.random.default_rng(0).integers(0, 256, (64, 96, 3), dtype=np.uint8))
smoke_operations = ["gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur", "motion_blur", "zoom_blur", "brightness", "contrast", "rotation", "zooming", "shifting"]
for operation in smoke_operations:
    result = apply_corruption(smoke_image, operation, 1, "smoke.png", 123)
    if result.size != smoke_image.size:
        raise RuntimeError(f"Corruption {operation} changed image dimensions.")
print(f"Corruption smoke test passed for {len(smoke_operations)} operations.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Official AF-CLIP commit: {resolved_commit}")


## AF-CLIP+ few-shot protocol

AF-CLIP+ does not have separately trained few-shot checkpoints. It loads the official MVTec/VisA prompt and adaptor components and builds a category-specific memory bank from clean normal images in the target dataset's training split. MVTec targets use the VisA-trained components; VisA targets use the MVTec-trained components. The selected support images remain clean and fixed across every test corruption.

The paper reports 1-, 2-, and 4-shot means over five random support selections but does not publish those seeds. This notebook defaults to one explicit reproducible support seed. Add five distinct values to `REFERENCE_SEEDS` to produce five independent archives per shot; seed-qualified names prevent overwrites. The official 518-pixel preprocessing, layers 6/12/18/24, 1/3/5 spatial aggregation, nearest-neighbour memory score, 0.1 prompt-score fusion, and Gaussian sigma 4 metric map are preserved.


In [ ]:
import gc

from few_shot.harness.dataset import AnomalyDetectionDataset, build_dataset_configs
from few_shot.harness.models import AFCLIPFewShotWrapper, discover_afclip_checkpoints
from few_shot.harness.runner import run_afclip_evaluations

# ==============================================================================
# USER-CONTROLLABLE DATASET AND PATH SETTINGS
# ==============================================================================
DATASET_NAME = "visa"  # "mvtec", "visa", or "both"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa", "both"}:
    raise ValueError("DATASET_NAME must be 'mvtec', 'visa', or 'both'.")
DATASETS_TO_RUN = ("mvtec", "visa") if DATASET_NAME == "both" else (DATASET_NAME,)
MVTEC_ROOT = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_ROOT = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
CLIP_DOWNLOAD_DIR = "/kaggle/working/clip-download"

dataset_configs = build_dataset_configs(mvtec_root=MVTEC_ROOT if "mvtec" in DATASETS_TO_RUN else None, visa_root=VISA_ROOT if "visa" in DATASETS_TO_RUN else None)
config_by_key = {config.name.lower(): config for config in dataset_configs}
for dataset_name in DATASETS_TO_RUN:
    key = "mvtec" if dataset_name == "mvtec" else "visa"
    config = next((item for item in dataset_configs if item.name.lower().startswith(key)), None)
    if config is None:
        raise FileNotFoundError(f"Dataset preflight could not resolve {dataset_name}. Edit its Kaggle path.")
    test_count = 0
    for category in config.categories:
        probe = AnomalyDetectionDataset(config=config, category=category)
        if not probe.samples:
            raise RuntimeError(f"No test samples for {config.name}/{category}.")
        missing_masks = [sample["sample_id"] for sample in probe.samples if sample.get("is_anomaly") and not (sample.get("mask_path") and Path(sample["mask_path"]).is_file())]
        if missing_masks:
            raise RuntimeError(f"Missing anomaly masks for {config.name}/{category}; first: {missing_masks[0]}")
        test_count += len(probe.samples)
    print(f"Dataset preflight passed: {config.name} ({test_count} test images).")

# ==============================================================================
# USER-CONTROLLABLE EXECUTION SETTINGS
# ==============================================================================
USE_CATEGORIZED_CORRUPTIONS = True
CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = ["gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur", "motion_blur", "zoom_blur", "brightness", "contrast"]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = CATEGORIZED_CORRUPTION_TYPES if USE_CATEGORIZED_CORRUPTIONS else UNCATEGORIZED_CORRUPTION_TYPES
INCLUDE_CLEAN_BASELINE = True
SEVERITY_LEVELS = [1, 2, 3, 4]
SHOTS_TO_RUN = [1, 2, 4]
REFERENCE_SEEDS = [111]  # Add four more explicit seeds for five trials.
DEVICE = "cuda"
BATCH_SIZE = 4
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"
STRICT_SOURCE_COMMIT = True

if not SHOTS_TO_RUN or set(SHOTS_TO_RUN) - {1, 2, 4}:
    raise ValueError("SHOTS_TO_RUN must contain values from [1, 2, 4].")
if not REFERENCE_SEEDS or len(set(REFERENCE_SEEDS)) != len(REFERENCE_SEEDS):
    raise ValueError("REFERENCE_SEEDS must be non-empty and unique.")
CHECKPOINT_PATHS = discover_afclip_checkpoints(str(AFCLIP_ROOT / "weight"))
print("Checkpoint preflight passed: released MVTec/VisA prompt and adaptor files.")

resolved_roots = {"mvtec" if config.name.lower().startswith("mvtec") else "visa": str(config.root_path) for config in dataset_configs}
support_probe = AFCLIPFewShotWrapper(checkpoint_paths=CHECKPOINT_PATHS, dataset_roots=resolved_roots, shot=max(SHOTS_TO_RUN), reference_seed=REFERENCE_SEEDS[0], device=DEVICE)
for dataset_name in DATASETS_TO_RUN:
    selections = support_probe._select_support_paths(dataset_name)
    print(f"Support preflight passed: {dataset_name}, {len(selections)} categories, {max(SHOTS_TO_RUN)} normal images/category.")

print(f"Launching {len(SHOTS_TO_RUN) * len(REFERENCE_SEEDS)} AF-CLIP+ run(s): shots={SHOTS_TO_RUN}, support seeds={REFERENCE_SEEDS}, datasets={DATASETS_TO_RUN}")
run_afclip_evaluations(mvtec_root=MVTEC_ROOT, visa_root=VISA_ROOT, output_root=OUTPUT_ROOT, afclip_root=str(AFCLIP_ROOT), checkpoint_paths=CHECKPOINT_PATHS, shots=SHOTS_TO_RUN, datasets=DATASETS_TO_RUN, reference_seeds=REFERENCE_SEEDS, device=DEVICE, batch_size=BATCH_SIZE, corruption_types=CORRUPTION_TYPES, severity_levels=SEVERITY_LEVELS, categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS, corruption_cache_root=CORRUPTION_CACHE_ROOT, corruption_cache_format=CORRUPTION_CACHE_FORMAT, corruption_seed=CORRUPTION_SEED, include_clean=INCLUDE_CLEAN_BASELINE, strict_source_commit=STRICT_SOURCE_COMMIT, clip_download_dir=CLIP_DOWNLOAD_DIR)

gc.collect()
torch.cuda.empty_cache()
archives = []
for shot in SHOTS_TO_RUN:
    for seed in REFERENCE_SEEDS:
        suffix = f"-seed-{seed}" if len(REFERENCE_SEEDS) > 1 else ""
        archives.append(f"AF-CLIP+-{shot}-shot{suffix}_artifacts.zip")
print(f"Finished. Collect {len(archives)} archive(s): {archives}")
